# Hyperparameter Tuning with Optuna (PyTorch)

This tutorial demonstrates how to use Optuna for hyperparameter optimization with kgcnn-torch models.
We load the ESOL solubility dataset, define an Optuna objective function that creates a SchNet model
with suggested hyperparameters, trains it using the `fit()` trainer, and returns the validation metric.

This tutorial demonstrates:
1. Loading the real ESOL dataset (1128 molecules)
2. Defining an Optuna objective function with kgcnn-torch
3. Creating and running an Optuna study (100 trials, 100 epochs each)
4. Visualizing optimization results
5. Retraining the best model

## 1. Dataset Preparation

We use the real ESOL dataset from MoleculeNet (1128 molecules with measured aqueous solubility).
The dataset is automatically downloaded and preprocessed.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

from kgcnn_torch.data.datasets.ESOLDataset import ESOLDataset

dataset = ESOLDataset()
print(f"ESOL dataset: {len(dataset)} molecules")
print(f"Example graph: {dataset[0]}")

## 2. Model Configuration

We use SchNet as our base model. The objective function will tune its hyperparameters.
SchNet expects PyG Data objects with atomic numbers (`z`), positions (`pos`), and edge connectivity.

In [ ]:
from kgcnn_torch.models.schnet import SchNetModel

# Test that the model works with our data
from torch_geometric.loader import DataLoader

test_model = SchNetModel(
    node_dim=32, depth=2, units=64,
    gauss_bins=20, gauss_distance=4.0,
    num_targets=1, make_distance=True, expand_distance=True
)
test_loader = DataLoader(dataset[:2], batch_size=2)
test_batch = next(iter(test_loader))
with torch.no_grad():
    out = test_model(test_batch)
print(f"Test output shape: {out.shape}")
del test_model

## 3. Optuna Objective Function

The objective function:
1. Suggests hyperparameters using the Optuna trial
2. Creates a SchNet model with those hyperparameters
3. Trains using `fit()` for 100 epochs
4. Returns the validation MAE metric

This matches the Keras tutorial's training budget (100 trials, 100 epochs, batch size 128).

In [ ]:
import optuna
from sklearn.model_selection import train_test_split
from kgcnn_torch.training.trainer import fit, eval_epoch

BATCH_SIZE = 128
EPOCHS = 100

# Pre-split data (same split for all trials)
all_indices = np.arange(len(dataset))
train_idx, val_idx = train_test_split(all_indices, test_size=0.25, random_state=42)

train_data = dataset[torch.tensor(train_idx).long()]
val_data = dataset[torch.tensor(val_idx).long()]

print(f"Train: {len(train_data)}, Val: {len(val_data)}")


def objective(trial):
    """Optuna objective function for hyperparameter optimization."""
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)

    # Suggest hyperparameters
    depth = trial.suggest_int("depth", 1, 6)
    units = trial.suggest_int("units", 32, 256, step=32)
    node_dim = trial.suggest_int("node_dim", 16, 128, step=16)
    gauss_bins = trial.suggest_int("gauss_bins", 10, 50, step=5)
    gauss_distance = trial.suggest_float("gauss_distance", 2.0, 8.0)
    node_pooling = trial.suggest_categorical("node_pooling", ["sum", "mean"])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)

    # Create model
    model = SchNetModel(
        node_dim=node_dim,
        depth=depth,
        units=units,
        gauss_bins=gauss_bins,
        gauss_distance=gauss_distance,
        gauss_sigma=0.4,
        node_pooling=node_pooling,
        last_mlp_units=[units, units // 2],
        num_targets=1,
        output_embedding="graph",
        make_distance=True,
        expand_distance=True
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Train
    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=torch.optim.Adam(model.parameters(), lr=learning_rate),
        loss_fn=nn.MSELoss(),
        epochs=EPOCHS,
        device=device,
        metrics={"mae": lambda pred, target: torch.mean(torch.abs(pred - target))},
        verbose=0
    )

    # Evaluate on validation set
    val_results = eval_epoch(
        model, val_loader,
        loss_fn=nn.MSELoss(),
        device=device,
        metrics={"mae": lambda pred, target: torch.mean(torch.abs(pred - target))}
    )

    return val_results["mae"]

## 4. Run Optimization

We create an Optuna study and optimize the objective function. The study explores
different combinations of hyperparameters and finds the configuration that minimizes
the validation MAE. We run 100 trials with a 10-minute timeout.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

study = optuna.create_study(direction="minimize")
study.optimize(
    objective,
    n_trials=100,
    timeout=600
)

## 5. Results

In [ ]:
print(f"Number of finished trials: {len(study.trials)}")
print(f"\nBest trial:")
trial = study.best_trial
print(f"  Value (MAE): {trial.value:.4f}")
print(f"  Params:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

In [ ]:
# Show all trial results
import pandas as pd

trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
print(trials_df.sort_values("value").head(10))

## 6. Visualization

Optuna provides built-in visualization tools to understand the optimization process.

In [ ]:
import matplotlib.pyplot as plt

# Plot optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial values over time
values = [t.value for t in study.trials if t.value is not None]
best_values = np.minimum.accumulate(values)

axes[0].plot(values, "o-", alpha=0.5, label="Trial MAE")
axes[0].plot(best_values, "r-", linewidth=2, label="Best MAE")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Validation MAE")
axes[0].set_title("Optimization History")
axes[0].legend()

# Parameter importance (simple approximation)
param_names = list(study.best_trial.params.keys())
param_values_all = {p: [] for p in param_names}
trial_values = []
for t in study.trials:
    if t.value is not None:
        trial_values.append(t.value)
        for p in param_names:
            if p in t.params:
                param_values_all[p].append(t.params[p])
            else:
                param_values_all[p].append(None)

# Correlation with objective for numeric params
correlations = []
corr_names = []
for p in param_names:
    vals = param_values_all[p]
    if all(isinstance(v, (int, float)) for v in vals if v is not None):
        numeric_vals = [v for v in vals if v is not None]
        corr_trial = [trial_values[i] for i, v in enumerate(vals) if v is not None]
        if len(numeric_vals) > 2:
            corr = abs(np.corrcoef(numeric_vals, corr_trial)[0, 1])
            correlations.append(corr if not np.isnan(corr) else 0)
            corr_names.append(p)

if correlations:
    axes[1].barh(corr_names, correlations, color="steelblue")
    axes[1].set_xlabel("|Correlation| with MAE")
    axes[1].set_title("Parameter Importance (correlation)")

plt.tight_layout()
plt.show()

In [ ]:
# Optuna's built-in visualization (if plotly is installed)
try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_param_importances,
        plot_contour
    )

    fig1 = plot_optimization_history(study)
    fig1.show()

    fig2 = plot_param_importances(study)
    fig2.show()

    fig3 = plot_contour(study, params=["depth", "learning_rate"])
    fig3.show()

except ImportError:
    print("Install plotly for interactive Optuna plots: pip install plotly")

## 7. Train Best Model

After the study finishes, we can train a final model using the best hyperparameters found.

In [ ]:
best_params = study.best_trial.params
print("Best hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# Build model with best params
best_model = SchNetModel(
    node_dim=best_params.get("node_dim", 64),
    depth=best_params["depth"],
    units=best_params["units"],
    gauss_bins=best_params.get("gauss_bins", 20),
    gauss_distance=best_params.get("gauss_distance", 4.0),
    gauss_sigma=0.4,
    node_pooling=best_params.get("node_pooling", "sum"),
    last_mlp_units=[best_params["units"], best_params["units"] // 2],
    num_targets=1,
    make_distance=True,
    expand_distance=True
)

# Full training with best hyperparameters
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

history = fit(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=torch.optim.Adam(
        best_model.parameters(),
        lr=best_params["learning_rate"]
    ),
    loss_fn=nn.MSELoss(),
    epochs=200,
    device=device,
    metrics={"mae": lambda pred, target: torch.mean(torch.abs(pred - target))},
    verbose=1
)

print(f"\nFinal train loss: {history['train_loss'][-1]:.4f}")
if history['val_loss']:
    print(f"Final val loss: {history['val_loss'][-1]:.4f}")
if 'val_mae' in history:
    print(f"Final val MAE: {history['val_mae'][-1]:.4f}")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(history["train_loss"], label="Train Loss")
if history["val_loss"]:
    ax.plot(history["val_loss"], label="Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training Curves (Best Model)")
ax.legend()
ax.set_yscale("log")
plt.show()

## Summary

This notebook demonstrated Optuna hyperparameter optimization with kgcnn-torch on the **real ESOL dataset** (1128 molecules):

1. **Dataset** -- Load ESOL via `ESOLDataset()` (auto-downloads and preprocesses)
2. **Objective function** -- Create SchNet model with `trial.suggest_*` params, train with `fit()`, return val MAE
3. **Study** -- `optuna.create_study().optimize()` explores the hyperparameter space (100 trials, 100 epochs, batch 128)
4. **Visualization** -- Plot optimization history and parameter importance
5. **Best model** -- Retrain with optimal hyperparameters found by Optuna

Key hyperparameters tuned:
- Model architecture: `depth`, `units`, `node_dim`, `node_pooling`
- Feature expansion: `gauss_bins`, `gauss_distance`
- Training: `learning_rate`